In [ ]:
!pip install --upgrade transformers==4.57.3 accelerate bitsandbytes

In [ ]:
!pip freeze > /kaggle/working/requirements.txt

## Import libraries

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split

from typing import Dict, List, Optional, Tuple

from datasets import Dataset

import torch
import torch.nn as nn
from torch.nn import functional as F

from transformers import (
    AutoTokenizer,
    AutoModel,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    TrainerCallback
)

import wandb
import huggingface_hub
from huggingface_hub import PyTorchModelHubMixin

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# I have saved my API token with "wandb_api" as Label. 
# If you use some other Label make sure to change the same below. 
wandb_api = user_secrets.get_secret("wandb_api") 

wandb.login(key=wandb_api)

huggingface_api = user_secrets.get_secret("huggingface_api") 
huggingface_hub.login(token=huggingface_api)

In [ ]:
print(torch.cuda.is_available())

In [ ]:
torch.cuda.is_bf16_supported()

## Set config

Set configurations for model

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Backbone model name
MODEL_NAME = "microsoft/deberta-v3-base"

# Max sequence length for input
MAX_SEQUENCE_LENGTH = 512

## Import dataset

The competition dataset comprises user interactions from the ChatBot Arena. In each interaction, a judge presents one or more prompts to two different large language models and then indicates which model provided the more satisfactory response. The training data contains `55,000` rows, with an expected `25,000` rows in the test set.

## Files

### `train.csv`
- `id`: Unique identifier for each row.
- `model_[a/b]`: Model identity, present in train.csv but not in test.csv.
- `prompt`: Input prompt given to both models.
- `response_[a/b]`: Model_[a/b]'s response to the prompt.
- `winner_model_[a/b/tie]`: Binary columns indicating the judge's selection (ground truth target).

### `test.csv`
- `id`: Unique identifier for each row.
- `prompt`: Input prompt given to both models.
- `response_[a/b]`: Model_[a/b]'s response to the prompt.

> Note that each interaction may have multiple prompts and responses, but this notebook will use only **one prompt per interaction**. You can choose to use all prompts and responses. Additionally, prompts and responses in the dataframe are provided as string-formatted lists, so they need to be converted to literal lists using `eval()`.


In [ ]:
raw_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")
test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

In [ ]:
raw_df.head()

In [ ]:
test_df.head()

## EDA

In [ ]:
# Check column data types
raw_df.dtypes

Count plot of different types models for the prompts in the data

In [ ]:
plt.figure(figsize=(12,5))
pd.concat([raw_df['model_a'], raw_df['model_b']]).value_counts().plot(kind='bar', stacked=True)

plt.show()

## Data pre-processing

Prompt and responses are in json format and need parsing

In [ ]:
import json

def safe_parse_json(x):
    if not isinstance(x, str):
        return x
    try:
        val = json.loads(x)
        # If it's a list, return first non-null element
        if isinstance(val, list):
            if val:
                return [item if item is not None else '' for item in val]
            else:
                return ''
        return val
    except json.JSONDecodeError:
        return ""

raw_df["response_a_processed"] = raw_df["response_a"].apply(safe_parse_json)
raw_df["response_b_processed"] = raw_df["response_b"].apply(safe_parse_json)
raw_df["prompt_processed"] = raw_df["prompt"].apply(safe_parse_json)

In [ ]:
test_df["response_a_processed"] = test_df["response_a"].apply(safe_parse_json)
test_df["response_b_processed"] = test_df["response_b"].apply(safe_parse_json)
test_df["prompt_processed"] = test_df["prompt"].apply(safe_parse_json)

### Contextualize Response with Prompt

In our approach, we will contextualize each response with the prompt instead of using a single prompt for all responses. This means that for each response, we will provide the model with the same set of prompts combined with their respective response (e.g., `(P + R_A)`, `(P + R_B)`, etc.). This approach is similar to the multiple-choice question task in NLP.

In [ ]:
def format_conversation(query_list, response_list):
    parts = []
    for i, (q, r) in enumerate(zip(query_list, response_list)):
        parts.append((f"Query:\n{q}\n\nResponse:\n{r}"))
    return '\n\n'.join(parts)

raw_df['text_a'] = raw_df.apply(lambda x: format_conversation(x['prompt_processed'], x['response_a_processed']), axis=1)
raw_df['text_b'] = raw_df.apply(lambda x: format_conversation(x['prompt_processed'], x['response_b_processed']), axis=1)

In [ ]:
test_df['text_a'] = test_df.apply(lambda x: format_conversation(x['prompt_processed'], x['response_a_processed']), axis=1)
test_df['text_b'] = test_df.apply(lambda x: format_conversation(x['prompt_processed'], x['response_b_processed']), axis=1)

### Summary statistics for the number of words in the conversation texts. 

This helps to determine the maximum input sequence length to the model and hence helps decide the model to choose.

In [ ]:
word_split = raw_df["text_a"].apply(lambda x: x.split(' '))
word_split.apply(lambda x: len(x)).describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.90])

In [ ]:
word_split = raw_df["text_b"].apply(lambda x: x.split(' '))
word_split.apply(lambda x: len(x)).describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.90])

> The conversations mostly have < 550 words in each conversation. Assuming $ \text{Tokens per conversation} = 1.5 \times \text{Words per conversation} $, we would ideally need a model which can handle ~850 tokens.

### Create target label

Create a single target column which can help determine the true class

In [ ]:
def create_target_col(encoding):
    """
    Create column for target labels
    """

    if encoding == [0, 0, 1]:
        return 'tie'
    elif encoding == [0, 1, 0]:
        return 'model_b'
    elif encoding == [1, 0, 0]:
        return 'model_a'

    return np.nan

raw_df['target'] = raw_df[['winner_model_a', 'winner_model_b', 'winner_tie']].apply(lambda x: create_target_col(list(x)), axis=1)

raw_df['label'] = raw_df['target'].map({'model_a': 0, 'model_b': 1, 'tie': 2})

## Train-test split

In [ ]:

train_df, eval_df = train_test_split(raw_df, test_size=0.2, random_state=42, stratify=raw_df["label"])

## DataLoader

The code below sets up a robust data flow pipeline using for data processing. We tokenize the text seqences and set-up the data in format acceptable by the model architecture

### Tokenize and format data

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Any
import torch

def tokenize_pairwise(batch):
    a_encodings = tokenizer(
        batch["text_a"],
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )
    b_encodings = tokenizer(
        batch["text_b"],
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )

    out_dict = {
        "a_input_ids": a_encodings["input_ids"],
        "a_attention_mask": a_encodings["attention_mask"],
        "b_input_ids": b_encodings["input_ids"],
        "b_attention_mask": b_encodings["attention_mask"]
    }

    if "label" in batch:
        out_dict["labels"] = batch["label"]
    
    return out_dict


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

In [ ]:
from datasets import Dataset

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

train_dataset = train_dataset.map(tokenize_pairwise, batched=True, remove_columns=list(train_df.columns))
eval_dataset = eval_dataset.map(tokenize_pairwise, batched=True, remove_columns=list(eval_df.columns))


In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["a_input_ids","a_attention_mask","b_input_ids","b_attention_mask","labels"]
)

eval_dataset.set_format(
    type="torch",
    columns=["a_input_ids","a_attention_mask","b_input_ids","b_attention_mask","labels"]
)

In [ ]:
## Steps to format test data

test_dataset = Dataset.from_pandas(test_df)

test_dataset = test_dataset.map(tokenize_pairwise, batched=True, remove_columns=list(test_df.columns))

test_dataset.set_format(
    type="torch",
    columns=["a_input_ids","a_attention_mask","b_input_ids","b_attention_mask"]
)

## Model Architecture

Our approach utilizes a Transformer Encoder Backbone to process each prompt and response pair, generating output embeddings. We then concatenate these embeddings and pass them through Pooling layers and a classifier to obtain logits, followed by a `softmax` function for the final output.

When dealing with multiple responses, we use a weight-sharing strategy. This means we provide the model with one response at a time along with the prompt `(P + R_A)`, `(P + R_B)`, etc., using the same model weights for all responses. After obtaining embeddings for all responses, we concatenate them and apply average pooling. Next, we use a `Linear/Dense` layers along with the `Softmax` function as the classifier for the final result. Providing all responses at once would increase text length and complicate model handling. Note that, in the classifier, we use 3 classes for `winner_model_a`, `winner_model_b`, and `draw` cases.

The diagram below illustrates this approach:

<div align="center">
    <img src="https://i.postimg.cc/g0gcvy3f/Kaggle-drawio.png">
</div>

From a coding perspective, note that we use the same model for all responses with shared weights, contrary to the separate models implied in the diagram.

### Metric

The metric for this competition is **Log Loss**. This metric can be expressed mathematically as,

$$
\text{Log Loss} = -\frac{1}{N} \sum_{i=1}^{N} \left( y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right)
$$

where $ N $ is the number of samples, $ y_i $ is the true label, and $ p_i $ is the predicted probability of the sample belonging to the positive class.

In [ ]:
from transformers import PreTrainedModel, PretrainedConfig, AutoModel
import inspect
from typing import Optional

def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    count = mask.sum(dim=1).clamp(min=1e-9)
    return summed / count

class PairwiseBiEncoderClassifierConfig(PretrainedConfig):
    model_type = "pairwise-biencoder-classifier"

    def __init__(
        self,
        base_model_name: Optional[str] = None,
        hidden_size: int = 768,
        num_labels: int = 3,
        dropout: float = 0.2,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.base_model_name = base_model_name
        self.hidden_size = hidden_size
        self.num_labels = num_labels
        self.hidden_dropout_prob = dropout

class PairwiseBiEncoderForSequenceClassification(PreTrainedModel):
    config_class = PairwiseBiEncoderClassifierConfig

    def __init__(self, config: PairwiseBiEncoderClassifierConfig, encoder: Optional[nn.Module] = None):
        """
        If `encoder` is provided, it will be used; otherwise AutoModel.from_pretrained(config.base_model_name)
        is used to construct the encoder.
        """
        super().__init__(config)
        # Use provided encoder (useful when you loaded quantized backbone) or build from config.base_model_name
        if encoder is not None:
            self.encoder = encoder
            # try to sync config.hidden_size if not already set
            if getattr(self.config, "hidden_size", None) is None and getattr(self.encoder, "config", None):
                self.config.hidden_size = self.encoder.config.hidden_size
        else:
            if not getattr(self.config, "base_model_name", None):
                raise ValueError("Either pass `encoder` or set config.base_model_name.")
            # load from base model name; allow user to pass quantization via from_pretrained external call if desired
            self.encoder = AutoModel.from_pretrained(self.config.base_model_name)

        hidden_size = self.config.hidden_size
        self.dropout = nn.Dropout(self.config.hidden_dropout_prob)

        # classifier sized from config.hidden_size and config.num_labels
        self.classifier = nn.Sequential(
            nn.Linear(4 * hidden_size, 2 * hidden_size),
            nn.GELU(),
            nn.Linear(2 * hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, self.config.num_labels),
        )

        # Initialize weights and apply final HF hooks
        self.post_init()

    # ----------------- pooling + encoding (same robust logic) -----------------
    def _encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)

        if isinstance(out, tuple):
            last_hidden_state = out[0]
            pooler_output = out[1] if len(out) > 1 else None
        else:
            last_hidden_state = getattr(out, "last_hidden_state", None)
            pooler_output = getattr(out, "pooler_output", None)

        if pooler_output is not None:
            return pooler_output
        if last_hidden_state is not None:
            # use the same mean_pool util you already have
            return mean_pool(last_hidden_state, attention_mask)

        raise ValueError("Encoder output missing last_hidden_state and pooler_output.")

    # ----------------- forward (compatible with HF Trainer / PEFT) -----------------
    def forward(
        self,
        a_input_ids: torch.Tensor,
        a_attention_mask: torch.Tensor,
        b_input_ids: torch.Tensor,
        b_attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        **kwargs,
    ):
        hA = self._encode(a_input_ids, a_attention_mask)
        hB = self._encode(b_input_ids, b_attention_mask)

        comb = torch.cat([hA, hB, torch.abs(hA - hB), hA * hB], dim=-1)
        logits = self.classifier(self.dropout(comb))

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.config.num_labels), labels.view(-1))

        # return HF-style outputs: for Trainer it's OK to return a dict with 'loss' and 'logits'
        return {"loss": loss, "logits": logits}

    # ----------------- freeze/unfreeze adapted for k-bit params -----------------
    def freeze_backbone(self):
        for name, param in self.encoder.named_parameters():
            dt = getattr(param, "dtype", None)
            if dt is None or not dt.is_floating_point:
                continue
            param.requires_grad = False

    def unfreeze_backbone(self):
        skipped = []
        for name, param in self.encoder.named_parameters():
            dt = getattr(param, "dtype", None)
            if dt is None or not dt.is_floating_point:
                skipped.append(name)
                continue
            param.requires_grad = True
        if skipped:
            print(f"Warning: skipped unfreezing {len(skipped)} non-float params (examples): {skipped[:6]}")

    def is_backbone_frozen(self) -> bool:
        float_params = [p for _, p in self.encoder.named_parameters() if getattr(p, "dtype", None) is not None and p.dtype.is_floating_point]
        return not any(p.requires_grad for p in float_params)

    # ---- place inside PairwiseBiEncoderForSequenceClassification ----

    def get_input_embeddings(self):
        """
        Return the input embeddings module. PEFT/transformers call this to
        enable input-require-grads behavior for gradient checkpointing.
        """
        # Try direct encoder, then encoder.base_model, then self.encoder (fallback)
        for cand in (getattr(self, "encoder", None), getattr(self.encoder, "base_model", None)):
            if cand is None:
                continue
            get_emb = getattr(cand, "get_input_embeddings", None)
            if callable(get_emb):
                return get_emb()
        # fallback: delegate to PreTrainedModel (will raise if not available)
        return super().get_input_embeddings()

    def set_input_embeddings(self, value):
        """
        Allow setting of the input embeddings module (delegated to encoder).
        """
        for cand in (getattr(self, "encoder", None), getattr(self.encoder, "base_model", None)):
            if cand is None:
                continue
            set_emb = getattr(cand, "set_input_embeddings", None)
            if callable(set_emb):
                return set_emb(value)
        return super().set_input_embeddings(value)

    def gradient_checkpointing_enable(self, **kwargs):
        """
        Delegate gradient_checkpointing_enable to the first module that supports it.
        Accepts arbitrary kwargs because Trainer may call it with
        gradient_checkpointing_kwargs=<dict>.
        """
        # Try the encoder itself first
        target_modules = []
        if hasattr(self, "encoder") and self.encoder is not None:
            target_modules.append(self.encoder)
        # fallback: try self too (in case wrappers attach the method here)
        target_modules.append(self)

        for module in target_modules:
            if hasattr(module, "gradient_checkpointing_enable"):
                fn = getattr(module, "gradient_checkpointing_enable")
                try:
                    # try passing kwargs (Trainer does this)
                    fn(**kwargs)
                    # print(f"[gradient_checkpointing_enable] called on {module} with kwargs={kwargs}")
                    return
                except TypeError:
                    # the underlying method doesn't accept kwargs -> call without them
                    fn()
                    # print(f"[gradient_checkpointing_enable] called on {module} without kwargs")
                    return

        # final fallback: search children (useful if it's inside a wrapper)
        for module in self.modules():
            if hasattr(module, "gradient_checkpointing_enable"):
                fn = getattr(module, "gradient_checkpointing_enable")
                try:
                    fn(**kwargs)
                    # print(f"[gradient_checkpointing_enable] called on child {module} with kwargs={kwargs}")
                    return
                except TypeError:
                    fn()
                    # print(f"[gradient_checkpointing_enable] called on child {module} without kwargs")
                    return

        # nothing found — no-op (or raise if you prefer)
        # print("[gradient_checkpointing_enable] no module found to enable gradient checkpointing")

    def gradient_checkpointing_disable(self, **kwargs):
        """
        Delegate gradient_checkpointing_disable similarly; accept kwargs for safety.
        """
        target_modules = []
        if hasattr(self, "encoder") and self.encoder is not None:
            target_modules.append(self.encoder)
        target_modules.append(self)

        for module in target_modules:
            if hasattr(module, "gradient_checkpointing_disable"):
                fn = getattr(module, "gradient_checkpointing_disable")
                try:
                    fn(**kwargs)
                    # print(f"[gradient_checkpointing_disable] called on {module} with kwargs={kwargs}")
                    return
                except TypeError:
                    fn()
                    # print(f"[gradient_checkpointing_disable] called on {module} without kwargs")
                    return

        for module in self.modules():
            if hasattr(module, "gradient_checkpointing_disable"):
                fn = getattr(module, "gradient_checkpointing_disable")
                try:
                    fn(**kwargs)
                    # print(f"[gradient_checkpointing_disable] called on child {module} with kwargs={kwargs}")
                    return
                except TypeError:
                    fn()
                    # print(f"[gradient_checkpointing_disable] called on child {module} without kwargs")
                    return

        # print("[gradient_checkpointing_disable] no module found to disable gradient checkpointing")


class FreezeUnfreezeCallback(TrainerCallback):
    """
    Freezes backbone for initial epochs, then unfreezes later.
    """

    def __init__(self, unfreeze_at_epoch: int = 1):
        self.unfreeze_at_epoch = unfreeze_at_epoch
        self.has_unfrozen = False

    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        if state.epoch < self.unfreeze_at_epoch:
            if not model.is_backbone_frozen():
                print(f"Epoch {int(state.epoch)}: Freezing backbone.")
                model.freeze_backbone()
        elif not self.has_unfrozen:
            print(f"Epoch {int(state.epoch)}: Unfreezing backbone.")
            model.unfreeze_backbone()
            self.has_unfrozen = True

In [ ]:
from dataclasses import dataclass
from typing import List, Any, Dict
import torch

@dataclass
class PairwiseDataCollator:
    tokenizer: Any
    padding: str = "longest"   # "longest" or "max_length"
    max_length: int = 128

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # defensive: filter out None
        features = [f for f in features if f is not None and isinstance(f, dict)]
        if len(features) == 0:
            raise ValueError("Empty batch after filtering None features in collator.")

        # ensure required keys exist
        for i, f in enumerate(features):
            for k in ("a_input_ids", "a_attention_mask", "b_input_ids", "b_attention_mask"):
                if k not in f:
                    raise ValueError(f"Missing key {k} in feature at batch pos {i}: keys={list(f.keys())}")

        a_feats = [{"input_ids": f["a_input_ids"], "attention_mask": f["a_attention_mask"]} for f in features]
        b_feats = [{"input_ids": f["b_input_ids"], "attention_mask": f["b_attention_mask"]} for f in features]

        # NOTE: truncation should have been done during tokenization (when creating the dataset).
        # pad() does not take `truncation` argument on many tokenizers, so we remove it here.
        pad_kwargs = {"padding": self.padding, "return_tensors": "pt"}
        if self.padding == "max_length":
            pad_kwargs["max_length"] = self.max_length

        a_batch = self.tokenizer.pad(a_feats, **pad_kwargs)
        b_batch = self.tokenizer.pad(b_feats, **pad_kwargs)

        out = {
            "a_input_ids": a_batch["input_ids"],
            "a_attention_mask": a_batch["attention_mask"],
            "b_input_ids": b_batch["input_ids"],
            "b_attention_mask": b_batch["attention_mask"],
        }

        # optional labels
        if "labels" in features[0]:
            labels = [int(f["labels"]) for f in features]
            out["labels"] = torch.tensor(labels, dtype=torch.long)

        # carry ids if present (not tensorized)
        if "id" in features[0]:
            out["ids"] = [f.get("id") for f in features]

        return out


## Model training strategy

We first train the custom model head while keeping the backbone model as is. Next, we use the LoRA + Qunatization to fine-tune the model efficiently

### Fine-tune the custom model head first

In [ ]:
backbone = AutoModel.from_pretrained(MODEL_NAME) #quantization_config=quant_config, device_map="auto"

hidden_size = backbone.config.hidden_size

assert MAX_SEQUENCE_LENGTH <= backbone.config.max_position_embeddings, f"Config 'max_sequence_length' must be <= the max sequence length allowed by the model i.e. {backbone.config.max_position_embeddings}"

# Build config (hidden_size will be inferred if you want)
cfg = PairwiseBiEncoderClassifierConfig(base_model_name=MODEL_NAME, hidden_size=backbone.config.hidden_size, num_labels=3)

# Create wrapper model using prebuilt backbone
model = PairwiseBiEncoderForSequenceClassification(cfg, encoder=backbone)

In [ ]:
from transformers import AutoConfig, AutoModel

AutoConfig.register("pairwise-biencoder-classifier", PairwiseBiEncoderClassifierConfig)
AutoModel.register(PairwiseBiEncoderClassifierConfig, PairwiseBiEncoderForSequenceClassification)

PairwiseBiEncoderClassifierConfig.register_for_auto_class()
PairwiseBiEncoderForSequenceClassification.register_for_auto_class("AutoModel")

In [ ]:
# find last checkpoint path (example)
from transformers.trainer_utils import get_last_checkpoint
import os

data_collator = PairwiseDataCollator(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="/kaggle/working/deberta-v3-base-pairwise-sequence-classifier/v20251212",
    num_train_epochs=2,
    per_device_train_batch_size=64,
    eval_strategy="epoch",
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to='wandb',
    push_to_hub=True,
    hub_model_id='bheshaj/deberta-v3-base-pairwise-sequence-classifier',
    hub_private_repo=False,
    save_strategy="epoch",
    save_total_limit=1,           # keep last n checkpoints (optional)
    load_best_model_at_end=True,    # optional
    save_safetensors=True,           # recommended
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    # callbacks=[FreezeUnfreezeCallback(unfreeze_at_epoch=3)],  # ⬅️ add callback here
)

if os.path.exists(training_args.output_dir):
    ckpt_path = get_last_checkpoint(training_args.output_dir)
else:
    ckpt_path = None

print("Resolved checkpoint path:", ckpt_path)

# If checkpoint found, disable gradient checkpointing on wrapped modules before resuming
if ckpt_path is not None:
    # prefer delegators on model (add these if you haven't already)
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    else:
        # try encoder and all submodules as fallback
        if hasattr(model, "encoder") and hasattr(model.encoder, "gradient_checkpointing_disable"):
            model.encoder.gradient_checkpointing_disable()
        else:
            for module in model.modules():
                if hasattr(module, "gradient_checkpointing_disable"):
                    module.gradient_checkpointing_disable()

    # sanity checks: list files in checkpoint
    print("Checkpoint files:", os.listdir(ckpt_path))
    # resume
    trainer.train(resume_from_checkpoint=ckpt_path)
else:
    print("No checkpoint found, starting fresh training.")
    trainer.train()

In [ ]:
trainer.push_to_hub()

## Fine-tune entire model using LoRA + Quantization

In this section, we fine-tune the **entire pretrained model** using a combination of **Low-Rank Adaptation (LoRA)** and **model quantization** to achieve parameter-efficient training at scale.

Instead of updating all model weights—which is computationally expensive and memory-intensive—we inject small trainable LoRA adapters into selected layers while keeping the original weights frozen. Quantization further reduces memory footprint by loading the base model in low-precision (e.g., 4-bit or 8-bit), enabling fine-tuning of large language models on limited hardware.

This approach provides:
- Significant reduction in GPU memory usage
- Faster training and lower compute cost
- Competitive performance compared to full fine-tuning
- Practical scalability for experimentation and deployment

By combining LoRA with quantization, we strike an effective balance between **training efficiency** and **model performance**, making full-model adaptation feasible even on smaller GPUs in Kaggle free tier.


### Quantization and LoRA config

In [ ]:
# Trained model name from above step
PAIRWISE_BIENCODER_MODEL_NAME = "bheshaj/deberta-v3-base-pairwise-sequence-classifier"

# Use quantization
USE_4BIT = True

# PEFT/LoRA Params
LORA_TARGETS = ["query_proj", "key_proj", "value_proj", "dense"]
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    load_in_8bit=not USE_4BIT,
    bnb_4bit_quant_type="nf4" if USE_4BIT else None,
    bnb_4bit_use_double_quant=True if USE_4BIT else None,
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_4BIT and torch.cuda.is_available() else None,
)

# Apply LoRA
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="SEQ_CLS",
    base_model_name_or_path = PAIRWISE_BIENCODER_MODEL_NAME,
)

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoConfig
config = AutoConfig.from_pretrained(PAIRWISE_BIENCODER_MODEL_NAME, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(PAIRWISE_BIENCODER_MODEL_NAME)
model = AutoModel.from_pretrained(PAIRWISE_BIENCODER_MODEL_NAME, config=config, trust_remote_code=True)

In [ ]:
quant_encoder = AutoModel.from_pretrained(
    model.config.base_model_name,
    quantization_config=quant_config,
    trust_remote_code=True,
)

model.encoder = quant_encoder

# Prepare backbone for k-bit training
model = prepare_model_for_kbit_training(model)

model.config._name_or_path = PAIRWISE_BIENCODER_MODEL_NAME
model.config.base_model_name_or_path = PAIRWISE_BIENCODER_MODEL_NAME

model.freeze_backbone()

# Prepare model for quantization and LoRA
model = get_peft_model(model, lora_cfg)

In [ ]:
model.print_trainable_parameters()

In [ ]:
print(model.targeted_module_names)

In [ ]:
# find last checkpoint path (example)
from transformers.trainer_utils import get_last_checkpoint
import os

data_collator = PairwiseDataCollator(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="/kaggle/working/deberta-v3-base-pairwise-sequence-classifier-peft-finetuned/v20251228",

    num_train_epochs=4,
    per_device_train_batch_size=80,
    eval_strategy="epoch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="wandb",
    push_to_hub=False,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    save_safetensors=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    # callbacks=[FreezeUnfreezeCallback(unfreeze_at_epoch=0)],  # ⬅️ add callback here
)

if os.path.exists(training_args.output_dir):
    ckpt_path = get_last_checkpoint(training_args.output_dir)
else:
    ckpt_path = None

print("Resolved checkpoint path:", ckpt_path)

# If checkpoint found, disable gradient checkpointing on wrapped modules before resuming
if ckpt_path is not None:
    # prefer delegators on model (add these if you haven't already)
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    else:
        # try encoder and all submodules as fallback
        if hasattr(model, "encoder") and hasattr(model.encoder, "gradient_checkpointing_disable"):
            model.encoder.gradient_checkpointing_disable()
        else:
            for module in model.modules():
                if hasattr(module, "gradient_checkpointing_disable"):
                    module.gradient_checkpointing_disable()

    # sanity checks: list files in checkpoint
    print("Checkpoint files:", os.listdir(ckpt_path))
    # resume
    trainer.train(resume_from_checkpoint=ckpt_path)
else:
    print("No checkpoint found, starting fresh training.")
    trainer.train()

In [ ]:
SAVE_DIR = "/kaggle/working/pairwise-biencoder-qlora-adapter"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

In [ ]:
TARGET_REPO = "bheshaj/deberta-v3-base-pairwise-sequence-classifier-peft-finetuned"

# Safety assertions
assert model.config._name_or_path == "bheshaj/deberta-v3-base-pairwise-sequence-classifier"
assert "peft" in TARGET_REPO or "qlora" in TARGET_REPO

from huggingface_hub import upload_folder

upload_folder(
    repo_id=TARGET_REPO,
    folder_path=SAVE_DIR,
)


## Prediction

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModel, AutoTokenizer, AutoConfig

ADAPTER_REPO = "bheshaj/deberta-v3-base-pairwise-sequence-classifier-peft-finetuned"

adapter_config = PeftConfig.from_pretrained(ADAPTER_REPO)
tokenizer = AutoTokenizer.from_pretrained(adapter_config.base_model_name_or_path)
base_model_config = AutoConfig.from_pretrained(adapter_config.base_model_name_or_path, trust_remote_code=True)
base_model = AutoModel.from_pretrained(adapter_config.base_model_name_or_path, config=base_model_config, trust_remote_code=True)

# Load the Lora model
inference_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)

In [ ]:
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader

dataloader = DataLoader(
    test_dataset,
    batch_size=16,        # or whatever fits GPU
    shuffle=False
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_logits = []

for step, batch in enumerate(tqdm(dataloader)):
    # batch.to(device)
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(
            a_input_ids=batch["a_input_ids"],
            a_attention_mask=batch["a_attention_mask"],
            b_input_ids=batch["b_input_ids"],
            b_attention_mask=batch["b_attention_mask"],
        )

    all_logits.append(outputs['logits'].cpu())

logits = torch.cat(all_logits, dim=0)
preds = logits.argmax(dim=-1)

## Future Directions

In this notebook, we've achieved a good score with a small base model and modest token length. But there's plenty of room to improve. Here's how:

1. Try bigger models like `Gemma`.
2. Increase max token length to reduce loss of data.
3. Use a five-fold cross-validation and ensemble to make the model robust and get better scores.
4. Add augmentation like shuffling response orders for more robust performance.
5. Train for more epochs.
6. Tune the learning rate scheduler.

## Reference

* [LMSYS: KerasNLP Starter](https://www.kaggle.com/code/addisonhoward/lmsys-kerasnlp-starter)